In [ ]:
def load_dataset(self, path="/content/drive/MyDrive/NLP_Project_Dataset/news_summary.csv", encoding='latin-1'):
    # Read CSV file
    df = pd.read_csv("/content/drive/MyDrive/NLP_Project_Dataset/news_summary.csv", encoding='latin-1')

    if "text" not in df.columns or "ctext" not in df.columns:
        raise ValueError("CSV must contain 'text' and 'ctext' columns.")

    # Drop all other columns by selecting only 'text' and 'ctext'
    df = df[["text", "ctext"]].copy()

    # Drop rows with missing values in either column
    df.dropna(subset=["text", "ctext"], inplace=True)

    # Reset index
    df.reset_index(drop=True, inplace=True)

    # Store as lists
    self.dataset = df["text"].astype(str).tolist()
    self.summaries = df["ctext"].astype(str).tolist()

    # Confirm columns after dropping
    print(f"Final columns in dataset: {list(df.columns)}")
    print(f"Loaded {len(df)} clean examples from '{path}'.")

pd.set_option('display.max_columns', None)  # Show all columns
print(df.head())  # Or print(df) for full DataFrame

# Keep only 'text' and 'ctext'
df=df[["text", "ctext"]].copy()
df.dropna(subset=["text", "ctext"], inplace=True)
print(df.head())

                                                text  \
0  The Administration of Union Territory Daman an...   
1  Malaika Arora slammed an Instagram user who tr...   
2  The Indira Gandhi Institute of Medical Science...   
3  Lashkar-e-Taiba's Kashmir commander Abu Dujana...   
4  Hotels in Maharashtra will train their staff t...   

                                               ctext  
0  The Daman and Diu administration on Wednesday ...  
1  From her special numbers to TV?appearances, Bo...  
2  The Indira Gandhi Institute of Medical Science...  
3  Lashkar-e-Taiba's Kashmir commander Abu Dujana...  
4  Hotels in Mumbai and other Indian cities are t...  
                                                text  \
0  The Administration of Union Territory Daman an...   
1  Malaika Arora slammed an Instagram user who tr...   
2  The Indira Gandhi Institute of Medical Science...   
3  Lashkar-e-Taiba's Kashmir commander Abu Dujana...   
4  Hotels in Maharashtra will train their staff t... 

In [ ]:
# Prompt user to select an index
num_rows = len(df)
index_input = int(input(f"\nEnter a number between 1 and {num_rows} to select a row: "))
if not (1 <= index_input <= num_rows):
    raise ValueError("Invalid selection. Please choose a number within the valid range.")

index = index_input - 1  # Adjust for 0-based indexing
source_text = df.loc[index, "text"]


# Rest of the transformer pipeline
!pip install transformers sentencepiece torch --quiet

from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from transformers import pipeline

# 1. Load translation model
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
translation_model = MBartForConditionalGeneration.from_pretrained(model_name)

# 2. Load summarization model
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# 3. Load sentiment analysis model
sentiment_model = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

# 4. Load NER model
ner_model = pipeline("ner", model="Davlan/bert-base-multilingual-cased-ner-hrl", aggregation_strategy="simple")

# Language selection
available_langs = {
    "1": {"name": "Hindi", "code": "hi_IN"},
    "2": {"name": "Tamil", "code": "ta_IN"},
    "3": {"name": "Bengali", "code": "bn_IN"},
    "4": {"name": "Malayalam", "code": "ml_IN"},
    "5": {"name": "Telugu", "code": "te_IN"},
    "6": {"name": "Marathi", "code": "mr_IN"},
    "7": {"name": "Gujarati", "code": "gu_IN"}
}

print("\nAvailable languages for translation:")
for key, lang in available_langs.items():
    print(f"{key}. {lang['name']}")

choice = input("\nEnter the number of your preferred language: ")
selected_lang_code = available_langs.get(choice, available_langs["1"])["code"]
selected_lang_name = available_langs.get(choice, available_langs["1"])["name"]

# Translation
tokenizer.src_lang = "en_XX"
encoded = tokenizer(source_text, return_tensors="pt")
generated_tokens = translation_model.generate(**encoded, forced_bos_token_id=tokenizer.lang_code_to_id[selected_lang_code])
translated_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

# Back Translation
tokenizer.src_lang = selected_lang_code
encoded_back = tokenizer(translated_text, return_tensors="pt")
back_tokens = translation_model.generate(**encoded_back, forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"])
back_translated_text = tokenizer.batch_decode(back_tokens, skip_special_tokens=True)[0]

# Summarization
english_summary = summarizer(source_text, max_length=50, min_length=10, do_sample=False)[0]['summary_text']

# Hindi summary (if chosen)
hindi_summary = ""
if selected_lang_name == "Hindi":
    tokenizer.src_lang = "en_XX"
    encoded_summary = tokenizer(english_summary, return_tensors="pt")
    generated_summary_tokens = translation_model.generate(**encoded_summary, forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"])
    hindi_summary = tokenizer.batch_decode(generated_summary_tokens, skip_special_tokens=True)[0]

# Sentiment
sentiment = sentiment_model(source_text)[0]

# NER
entities = ner_model(source_text)

# Output
print("\n--- Original Text (English) ---")
print(source_text)

print(f"\n--- Translation ({selected_lang_name}) ---")
print(translated_text)

print("\n--- Back Translated Text (to English) ---")
print(back_translated_text)

print("\n--- Summarized Text (English) ---")
print(english_summary)

if hindi_summary:
    print("\n--- Summarized Text (Hindi) ---")
    print(hindi_summary)

print("\n--- Sentiment ---")
print(f"Label: {sentiment['label']}, Score: {round(sentiment['score'], 2)}")

print("\n--- Named Entities ---")
for ent in entities:
    print(f"{ent['word']} ({ent['entity_group']})")



Enter a number between 1 and 4396 to select a row: 45


Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0



Available languages for translation:
1. Hindi
2. Tamil
3. Bengali
4. Malayalam
5. Telugu
6. Marathi
7. Gujarati

Enter the number of your preferred language: 6

--- Original Text (English) ---
Finance Minister Arun Jaitley has said that a "proper decision" would be taken on Air India's future and it would be in government's interest to protect jobs at the airline. Noting that the airline has debt of over ?50,000 crore, he said it is "not small" and "now we have to decide what has to be done with Air India".

--- Translation (Marathi) ---
अर ् थशास ् त ् र मंत ् री आर ् ण जेट ् ली म ् हणाल ् या आहेत अर ् ट इंडीयाच ् या भविष ् याबद ् दल "उत ् तम निर ् णय" घेण ् यास आणि या कंपनीतल ् या कामगारांचे सुरक ् षित करण ् याची सरकारच ् या हितात आहे. या कंपनीच ् या ५०,००० கோடி च ् या कर ् जाची लक ् ष वेधून ते म ् हणाल ् या आहेत ते "छोटं नाही" आणि "आता आम ् हाला ठरवायचं आहे अर ् ट इंडीयाबरोबर काय करायचं आहे ते".

--- Back Translated Text (to English) ---
The Minister of Economics, R.J., has said, "

In [ ]:
!pip install transformers sentencepiece torch --quiet

from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from transformers import pipeline

# 1. Load translation model
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
translation_model = MBartForConditionalGeneration.from_pretrained(model_name)

# 2. Load summarization model
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# 3. Load sentiment analysis model
sentiment_model = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")

# 4. Load NER model
ner_model = pipeline("ner", model="Davlan/bert-base-multilingual-cased-ner-hrl", aggregation_strategy="simple")

# ---- INPUT ----
source_text = "Climate change refers to long-term shifts in temperatures and weather patterns, mainly caused by human activities, especially the burning of fossil fuels. These shifts may be natural, such as through variations in the solar cycle. But since the 1800s, human activities have been the main driver of climate change, primarily due to burning coal, oil, and gas. Burning these materials releases what are called greenhouse gases into Earth's atmosphere. These emissions have risen to levels the planet has not seen in millions of years. As a result, the climate system is now changing faster than at any point in the history of modern civilization. The impacts of climate change are already being felt, from rising sea levels and melting glaciers to more intense heatwaves and extreme weather events."
source_lang = "en_XX"
target_langs = {
    "Hindi": "hi_IN",
    "Tamil": "ta_IN",
    "Bengali": "bn_IN",
    "Malayalam": "ml_IN",
    "Telugu": "te_IN",
    "Marathi": "mr_IN",
    "Gujarati": "gu_IN"
}

# ---- Translate to Multiple Languages ----
translations = {}
for lang_name, lang_code in target_langs.items():
    tokenizer.src_lang = source_lang
    encoded = tokenizer(source_text, return_tensors="pt")
    generated_tokens = translation_model.generate(**encoded, forced_bos_token_id=tokenizer.lang_code_to_id[lang_code])
    translated_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    translations[lang_name] = translated_text

# ---- Back Translation for all languages ----
back_translations = {}
for lang_name, lang_code in target_langs.items():
    tokenizer.src_lang = lang_code
    encoded_back = tokenizer(translations[lang_name], return_tensors="pt")
    back_tokens = translation_model.generate(**encoded_back, forced_bos_token_id=tokenizer.lang_code_to_id[source_lang])
    back_translated_text = tokenizer.batch_decode(back_tokens, skip_special_tokens=True)[0]
    back_translations[lang_name] = back_translated_text

# ---- Summarization ----
summary = summarizer(source_text, max_length=50, min_length=10, do_sample=False)[0]['summary_text']

# ---- Sentiment Analysis ----
sentiment = sentiment_model(source_text)[0]

# ---- Named Entity Recognition ----
entities = ner_model(source_text)

# ---- OUTPUT ----
print("\n--- Original Text ---")
print(source_text)

print("\n--- Translations ---")
for lang, text in translations.items():
    print(f"\n{lang}: {text}")

print("\n--- Back Translated Text (All languages → English) ---")
for lang, text in back_translations.items():
    print(f"\n{lang} → English: {text}")

print("\n--- Summarized Text ---")
print(summary)

print("\n--- Sentiment ---")
print(f"Label: {sentiment['label']}, Score: {round(sentiment['score'], 2)}")

print("\n--- Named Entities ---")
for ent in entities:
    print(f"{ent['word']} ({ent['entity_group']})")

Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0



--- Original Text ---
Climate change refers to long-term shifts in temperatures and weather patterns, mainly caused by human activities, especially the burning of fossil fuels. These shifts may be natural, such as through variations in the solar cycle. But since the 1800s, human activities have been the main driver of climate change, primarily due to burning coal, oil, and gas. Burning these materials releases what are called greenhouse gases into Earth's atmosphere. These emissions have risen to levels the planet has not seen in millions of years. As a result, the climate system is now changing faster than at any point in the history of modern civilization. The impacts of climate change are already being felt, from rising sea levels and melting glaciers to more intense heatwaves and extreme weather events.

--- Translations ---

Hindi: जलवायु परिवर्तन तापमान और मौसम के पैटर्न में दीर्घकालिक परिवर्तनों को निर्दिष्ट करता है, मुख्य रूप से मानव गतिविधियों के कारण, विशेष रूप से खनिज ईंधन 

In [ ]:
# ---- INPUT ----
source_text = "Climate change refers to long-term shifts in temperatures and weather patterns, mainly caused by human activities, especially the burning of fossil fuels. These shifts may be natural, such as through variations in the solar cycle. But since the 1800s, human activities have been the main driver of climate change, primarily due to burning coal, oil, and gas. Burning these materials releases what are called greenhouse gases into Earth's atmosphere. These emissions have risen to levels the planet has not seen in millions of years. As a result, the climate system is now changing faster than at any point in the history of modern civilization. The impacts of climate change are already being felt, from rising sea levels and melting glaciers to more intense heatwaves and extreme weather events."
source_lang = "en_XX"

# Dictionary of available languages
available_langs = {
    "1": {"name": "Hindi", "code": "hi_IN"},
    "2": {"name": "Tamil", "code": "ta_IN"},
    "3": {"name": "Bengali", "code": "bn_IN"},
    "4": {"name": "Malayalam", "code": "ml_IN"},
    "5": {"name": "Telugu", "code": "te_IN"},
    "6": {"name": "Marathi", "code": "mr_IN"},
    "7": {"name": "Gujarati", "code": "gu_IN"}
}

# Display available languages
print("Available languages for translation:")
for key, lang_info in available_langs.items():
    print(f"{key}. {lang_info['name']}")

# Get user input for language choice
choice = input("\nEnter the number of your preferred language: ")

# Validate user input
if choice in available_langs:
    selected_lang_name = available_langs[choice]["name"]
    selected_lang_code = available_langs[choice]["code"]
else:
    print("Invalid choice. Defaulting to Hindi.")
    selected_lang_name = "Hindi"
    selected_lang_code = "hi_IN"

print(f"\nTranslating to {selected_lang_name}...")

# ---- Translate to Selected Language ----
tokenizer.src_lang = source_lang
encoded = tokenizer(source_text, return_tensors="pt")
generated_tokens = translation_model.generate(**encoded, forced_bos_token_id=tokenizer.lang_code_to_id[selected_lang_code])
translated_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

# ---- Back Translation to English ----
tokenizer.src_lang = selected_lang_code
encoded_back = tokenizer(translated_text, return_tensors="pt")
back_tokens = translation_model.generate(**encoded_back, forced_bos_token_id=tokenizer.lang_code_to_id[source_lang])
back_translated_text = tokenizer.batch_decode(back_tokens, skip_special_tokens=True)[0]

# ---- Summarization in English ----
english_summary = summarizer(source_text, max_length=50, min_length=10, do_sample=False)[0]['summary_text']

# ---- Hindi Summarization (translate English summary to Hindi) ----
hindi_summary = ""
if selected_lang_name == "Hindi":
    tokenizer.src_lang = source_lang
    encoded_summary = tokenizer(english_summary, return_tensors="pt")
    generated_summary_tokens = translation_model.generate(**encoded_summary, forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"])
    hindi_summary = tokenizer.batch_decode(generated_summary_tokens, skip_special_tokens=True)[0]

    # Also generate a summary directly from the Hindi translation
    # Translating English summary is different from summarizing Hindi text directly
    # But since mBART is not trained for summarization in Hindi, we translate the English summary

# ---- Sentiment Analysis ----
sentiment = sentiment_model(source_text)[0]

# ---- Named Entity Recognition ----
entities = ner_model(source_text)

# ---- OUTPUT ----
print("\n--- Original Text (English) ---")
print(source_text)

print(f"\n--- Translation ({selected_lang_name}) ---")
print(translated_text)

print("\n--- Back Translated Text (to English) ---")
print(back_translated_text)

print("\n--- Summarized Text (English) ---")
print(english_summary)

if selected_lang_name == "Hindi":
    print("\n--- Summarized Text (Hindi) ---")
    print(hindi_summary)

print("\n--- Sentiment ---")
print(f"Label: {sentiment['label']}, Score: {round(sentiment['score'], 2)}")

print("\n--- Named Entities ---")
for ent in entities:
    print(f"{ent['word']} ({ent['entity_group']})")

Available languages for translation:
1. Hindi
2. Tamil
3. Bengali
4. Malayalam
5. Telugu
6. Marathi
7. Gujarati

Enter the number of your preferred language: 5

Translating to Telugu...

--- Original Text (English) ---
Climate change refers to long-term shifts in temperatures and weather patterns, mainly caused by human activities, especially the burning of fossil fuels. These shifts may be natural, such as through variations in the solar cycle. But since the 1800s, human activities have been the main driver of climate change, primarily due to burning coal, oil, and gas. Burning these materials releases what are called greenhouse gases into Earth's atmosphere. These emissions have risen to levels the planet has not seen in millions of years. As a result, the climate system is now changing faster than at any point in the history of modern civilization. The impacts of climate change are already being felt, from rising sea levels and melting glaciers to more intense heatwaves and extreme 

In [ ]:

# Supported languages dictionary
target_langs = {
    "Hindi": "hi_IN",
    "Tamil": "ta_IN",
    "Bengali": "bn_IN",
    "Malayalam": "ml_IN",
    "Telugu": "te_IN",
    "Marathi": "mr_IN",
    "Gujarati": "gu_IN"
}

source_lang = "en_XX"

# User inputs
print("\nAvailable target languages:", ', '.join(target_langs.keys()))
user_target = input("Choose a target language from the above list: ").strip()

if user_target not in target_langs:
    print("Invalid language selected. Exiting.")
    exit()

user_text = input("\nEnter the text you want to process: ")

# ---- Translate user text ----
tokenizer.src_lang = source_lang
encoded = tokenizer(user_text, return_tensors="pt")
generated_tokens = translation_model.generate(
    **encoded, forced_bos_token_id=tokenizer.lang_code_to_id[target_langs[user_target]]
)
translated_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

# ---- Back Translation to English ----
tokenizer.src_lang = target_langs[user_target]
encoded_back = tokenizer(translated_text, return_tensors="pt")
back_tokens = translation_model.generate(
    **encoded_back, forced_bos_token_id=tokenizer.lang_code_to_id[source_lang]
)
back_translated_text = tokenizer.batch_decode(back_tokens, skip_special_tokens=True)[0]

# ---- Summarization ----
summary = summarizer(user_text, max_length=50, min_length=10, do_sample=False)[0]['summary_text']

# ---- Sentiment Analysis ----
sentiment = sentiment_model(user_text)[0]

# ---- Named Entity Recognition ----
entities = ner_model(user_text)

# ---- Output ----
print("\n--- Original Text ---")
print(user_text)

print(f"\n--- Translated Text ({user_target}) ---")
print(translated_text)

print(f"\n--- Back Translated Text ({user_target} → English) ---")
print(back_translated_text)

print("\n--- Summarized Text ---")
print(summary)

print("\n--- Sentiment Analysis ---")
print(f"Label: {sentiment['label']}, Score: {round(sentiment['score'], 2)}")

print("\n--- Named Entities ---")
for ent in entities:
    print(f"{ent['word']} ({ent['entity_group']})")


Available target languages: Hindi, Tamil, Bengali, Malayalam, Telugu, Marathi, Gujarati
Choose a target language from the above list: Hindi

Enter the text you want to process: The rapid advancement of artificial intelligence (AI) is transforming the world in ways previously thought to be the domain of science fiction. From autonomous vehicles navigating complex cityscapes to virtual assistants responding to voice commands with uncanny accuracy, AI has permeated nearly every aspect of modern life. In healthcare, AI-powered diagnostic systems are helping doctors detect diseases earlier and with greater precision. In education, personalized learning platforms driven by machine learning algorithms are tailoring content to each student's pace and style of learning. Meanwhile, the business sector is experiencing a revolution through predictive analytics, chatbots, and robotic process automation, dramatically increasing efficiency and reducing costs.  However, this technological progress is